<a href="https://colab.research.google.com/github/faria392/telco-churn-xgboost/blob/main/Telco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Churn Prediction


## 1. Carregar os dados

In [ ]:
import pandas as pd
import numpy as np

dados = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
dados.head()

## 2. Exploração inicial

In [ ]:
print(f"Dimensões: {dados.shape[0]} linhas x {dados.shape[1]} colunas")
print(f"Linhas duplicadas: {dados.duplicated().sum()}")
print(f"Uso de memória: {dados.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
resumo = pd.DataFrame({
    "dtype": dados.dtypes,
    "nulos": dados.isnull().sum(),
    "% nulos": (dados.isnull().mean() * 100).round(2),
    "valores_unicos": dados.nunique(),
    "exemplo": dados.iloc[0]
})
resumo

In [ ]:
for col in dados.select_dtypes(include="object").columns:
    amostra = dados[col].astype(str).str.strip()
    convertida = pd.to_numeric(amostra, errors="coerce")
    n_falhas = convertida.isna().sum() - dados[col].isna().sum()
    if n_falhas > 0 and convertida.notna().sum() > 0:
        print(f"- '{col}': {n_falhas} valor(es) não numérico(s)/vazio(s) "
              f"que impedem conversão direta para número")

In [ ]:
dados.describe().T

In [ ]:
dados.describe(include="object").T

In [ ]:
for col in dados.select_dtypes(include="object").columns:
    valores = dados[col].unique()
    if len(valores) <= 10:
        print(f"- {col} ({len(valores)}): {list(valores)}")
    else:
        print(f"- {col} ({len(valores)}): alta cardinalidade, ex: {list(valores[:5])}...")

## 3. Limpeza (sem encoding isso fica só dentro do pipeline mais adiante)

Corrige `TotalCharges`: remove espaços em branco e converte para numérico.

In [ ]:
dados["TotalCharges"] = dados["TotalCharges"].astype(str).str.strip()
dados["TotalCharges"] = pd.to_numeric(dados["TotalCharges"], errors="coerce")
print(f"Valores nulos após conversão: {dados['TotalCharges'].isnull().sum()}")

Confirma o padrão das linhas afetadas — o problema ocorre justamente onde `tenure = 0`.

In [ ]:
dados[dados["TotalCharges"].isnull()][["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

Como `tenure = 0` significa que o cliente ainda não fechou um ciclo de cobrança, faz sentido preencher `TotalCharges` com 0 (não com média/mediana).

In [ ]:
dados["TotalCharges"] = dados["TotalCharges"].fillna(0)

`customerID` é apenas identificador único por linha — não deve entrar como feature no modelo.

In [ ]:
customer_ids = dados["customerID"]
dados = dados.drop(columns=["customerID"])

Padroniza as categorias redundantes `"No internet service"` e `"No phone service"` para `"No"`.

In [ ]:
cols_internet = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]
dados[cols_internet] = dados[cols_internet].replace("No internet service", "No")
dados["MultipleLines"] = dados["MultipleLines"].replace("No phone service", "No")

for col in cols_internet + ["MultipleLines"]:
    print(f"{col}: {dados[col].unique()}")

Apenas o alvo (`Churn`) precisa virar numérico agora; as demais colunas categóricas ficam como texto de propósito — o `ColumnTransformer` cuida delas mais adiante.

In [ ]:
dados["Churn"] = dados["Churn"].map({"Yes": 1, "No": 0})

print(f"Dimensões finais: {dados.shape}")
print(f"Nulos restantes:\n{dados.isnull().sum()[dados.isnull().sum() > 0]}")
print(f"\nTipos de dados:\n{dados.dtypes}")

## 4. Split treino (70%) / validação (15%) / teste (15%) / estratificado

In [ ]:
X = dados.drop(columns=["Churn"])
y = dados["Churn"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Treino:     {X_train.shape[0]:>5} amostras ({X_train.shape[0]/len(X):.1%}) | Churn=1: {y_train.mean():.2%}")
print(f"Validação:  {X_val.shape[0]:>5} amostras ({X_val.shape[0]/len(X):.1%}) | Churn=1: {y_val.mean():.2%}")
print(f"Teste:      {X_test.shape[0]:>5} amostras ({X_test.shape[0]/len(X):.1%}) | Churn=1: {y_test.mean():.2%}")

## 5. Pipeline de pré-processamento (todo o encoding acontece aqui, sem leakage)

Ajuste estas listas se o conjunto de features mudar. `cat_alta_cardinalidade` e `cols_temporais` ficam vazias porque não se aplicam a este dataset.

In [ ]:
num_continuas = ["tenure", "MonthlyCharges", "TotalCharges"]

cat_baixa_cardinalidade = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "InternetService", "Contract",
    "PaperlessBilling", "PaymentMethod"
]

cat_alta_cardinalidade = []
cols_temporais = []

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

pipeline_numerica = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

pipeline_cat_baixa = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

transformers = [
    ("num", pipeline_numerica, num_continuas),
    ("cat_baixa", pipeline_cat_baixa, cat_baixa_cardinalidade),
]

if cat_alta_cardinalidade:
    from category_encoders import TargetEncoder
    pipeline_cat_alta = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Desconhecido")),
        ("encoder", TargetEncoder(smoothing=0.3))
    ])
    transformers.append(("cat_alta", pipeline_cat_alta, cat_alta_cardinalidade))

preprocessador = ColumnTransformer(transformers=transformers, remainder="drop")

## 6. Pipeline final com XGBoost

In [ ]:
from xgboost import XGBClassifier

pipeline_final = Pipeline(steps=[
    ("preprocessamento", preprocessador),
    ("modelo", XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        random_state=42,
        n_jobs=-1
    ))
])

## 7. Treinar e avaliar (ROC-AUC na validação)

In [ ]:
from sklearn.metrics import roc_auc_score

pipeline_final.fit(X_train, y_train)

proba_val = pipeline_final.predict_proba(X_val)[:, 1]
print(f"ROC-AUC (validação): {roc_auc_score(y_val, proba_val):.4f}")

Espaço de busca dos hiperparâmetros: `learning_rate` entre 0.01 e 0.30, `subsample` e `colsample_bytree` entre 0.6 e 1.0.

In [ ]:
from scipy.stats import randint, uniform

espaco_busca = {
    "modelo__n_estimators": randint(100, 600),
    "modelo__max_depth": randint(3, 10),
    "modelo__learning_rate": uniform(0.01, 0.29),
    "modelo__subsample": uniform(0.6, 0.4),
    "modelo__colsample_bytree": uniform(0.6, 0.4),
    "modelo__min_child_weight": randint(1, 10),
    "modelo__gamma": uniform(0, 0.5),
    "modelo__reg_alpha": uniform(0, 1),
    "modelo__reg_lambda": uniform(0, 2),
}

Roda a busca com `n_iter=50` combinações testadas via validação cruzada estratificada.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

cv_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

busca = RandomizedSearchCV(
    estimator=pipeline_final,
    param_distributions=espaco_busca,
    n_iter=50,
    scoring="roc_auc",
    cv=cv_estratificado,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

busca.fit(X_train, y_train)

print(f"Melhor ROC-AUC (média na validação cruzada): {busca.best_score_:.4f}")
print(f"\nMelhores hiperparâmetros:\n{busca.best_params_}")

In [ ]:
from sklearn.metrics import roc_auc_score

melhor_modelo = busca.best_estimator_

proba_val = melhor_modelo.predict_proba(X_val)[:, 1]
auc_val = roc_auc_score(y_val, proba_val)

print(f"ROC-AUC (validação, modelo antigo sem tuning): 0.8138")
print(f"ROC-AUC (validação, modelo com tuning):        {auc_val:.4f}")
print(f"Ganho: {(auc_val - 0.8138):+.4f}")

O modelo já tunado e treinado (via `RandomizedSearchCV`) é o modelo final — não se re-treina nada aqui, apenas reaproveitamos o que já foi ajustado no treino.

In [ ]:
modelo_final = busca.best_estimator_

Avalia no conjunto de teste usando o threshold padrão de 0.5.

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

proba_test = modelo_final.predict_proba(X_test)[:, 1]
pred_test = modelo_final.predict(X_test)

auc_test = roc_auc_score(y_test, proba_test)
print(f"ROC-AUC (teste): {auc_test:.4f}")
print(f"\nComparativo:")
print(f"  Validação (holdout): 0.8485")
print(f"  Teste:{auc_test:.4f}")
print(f"  Diferença:{(auc_test - 0.8485):+.4f}")

In [ ]:
print(classification_report(y_test, pred_test, target_names=["Não Churn", "Churn"]))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, pred_test,
    display_labels=["Não Churn", "Churn"],
    cmap="Blues",
    ax=ax
)
ax.set_title("Matriz de Confusão — Conjunto de Teste")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, proba_test, ax=ax, name="XGBoost (tunado)")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatório")
ax.set_title("Curva ROC — Conjunto de Teste")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay
import matplotlib.pyplot as plt

precisions, recalls, thresholds = precision_recall_curve(y_test, proba_test)

fig, ax = plt.subplots(figsize=(6, 5))
PrecisionRecallDisplay.from_predictions(y_test, proba_test, ax=ax, name="XGBoost (tunado)")
ax.set_title("Curva Precision-Recall — Conjunto de Teste")
plt.tight_layout()
plt.show()

Calcula o F1-score para cada threshold testado.

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, precisions[:-1], label="Precision", linewidth=2)
ax.plot(thresholds, recalls[:-1], label="Recall", linewidth=2)

f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
ax.plot(thresholds, f1_scores, label="F1-score", linewidth=2, linestyle="--")

ax.axvline(x=0.5, color="gray", linestyle=":", label="Threshold padrão (0.5)")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision, Recall e F1 por Threshold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
resultados = []
for t in np.arange(0.1, 0.91, 0.05):
    pred_t = (proba_test >= t).astype(int)
    from sklearn.metrics import precision_score, recall_score, f1_score
    resultados.append({
        "threshold": round(t, 2),
        "precision_churn": precision_score(y_test, pred_t, pos_label=1),
        "recall_churn": recall_score(y_test, pred_t, pos_label=1),
        "f1_churn": f1_score(y_test, pred_t, pos_label=1),
        "qtd_alertas": pred_t.sum()
    })

tabela_thresholds = pd.DataFrame(resultados)
tabela_thresholds